# Validadores customizados com anotações

Vimos no curso como anexar um objeto `Field` direto a um campo:

In [1]:
from pydantic import BaseModel, Field, ValidationError

class Model(BaseModel):
    number: int = Field(gt=0, lt=5)

Ou com um tipo anotado:

In [2]:
from typing import Annotated

BoundedInt = Annotated[int, Field(gt=0, lt=5)]

class Model(BaseModel):
    number: BoundedInt

Podemos fazer algo parecido com validadores.

Primeiro definimos a função de validação. Como ela fica **fora** de uma classe, é uma função comum, não um método de classe (não
precisamos do argumento `cls`).

Vamos ao exemplo do datetime, começando pelo validador before:

In [3]:
from datetime import datetime
from typing import Any

from dateutil.parser import parse

def parse_datetime(value: Any):
    if isinstance(value, str):
        try:
            return parse(value)
        except Exception as ex:
            raise ValueError(str(ex))
    return value

Agora queremos anexá-la, como validador before, a um tipo datetime, via anotação:

In [4]:
from pydantic import BeforeValidator

In [5]:
DateTime = Annotated[datetime, BeforeValidator(parse_datetime)]

Podemos usar esse tipo anotado em qualquer modelo:

In [6]:
class Model(BaseModel):
    dt: DateTime

In [7]:
print(f'Model(dt="2020/1/1 3pm"): {Model(dt="2020/1/1 3pm")!r}')

Model(dt="2020/1/1 3pm"): Model(dt=datetime.datetime(2020, 1, 1, 15, 0))


Também podemos adicionar o validador after, do mesmo jeito, com anotações.

In [8]:
from pydantic import AfterValidator

In [9]:
import pytz

def make_utc(dt: datetime) -> datetime:
    if dt.tzinfo is None:
        dt = pytz.utc.localize(dt)
    else:
        dt = dt.astimezone(pytz.utc)
    return dt

In [10]:
DateTimeUTC = Annotated[datetime, BeforeValidator(parse_datetime), AfterValidator(make_utc)]

E usamos em qualquer modelo:

In [11]:
class Model(BaseModel):
    dt: DateTimeUTC

In [12]:
print(f'Model(dt="2020/1/1 3pm"): {Model(dt="2020/1/1 3pm")!r}')

Model(dt="2020/1/1 3pm"): Model(dt=datetime.datetime(2020, 1, 1, 15, 0, tzinfo=<UTC>))


In [13]:
eastern = pytz.timezone('US/Eastern')
dt = eastern.localize(datetime(2020, 1, 1, 3, 0, 0))

print(f"Model(dt=dt): {Model(dt=dt)!r}")

Model(dt=dt): Model(dt=datetime.datetime(2020, 1, 1, 8, 0, tzinfo=<UTC>))


Também dá para ter vários validadores before e after, executados nesta ordem:
- before: da **direita para a esquerda**
- validador do Pydantic
- after: da **esquerda para a direita**

(A ordem é a da listagem na anotação — o equivalente de "baixo para cima / cima para baixo" dos decoradores.)

In [14]:
def before_validator_1(value):
    print("before_validator_1")
    return value

def before_validator_2(value):
    print("before_validator_2")
    return value

def before_validator_3(value):
    print("before_validator_3")
    return value

def after_validator_1(value):
    print("after_validator_1")
    return value

def after_validator_2(value):
    print("after_validator_2")
    return value

def after_validator_3(value):
    print("after_validator_3")
    return value

In [15]:
CustomType = Annotated[
    int,
    BeforeValidator(before_validator_1),
    AfterValidator(after_validator_1),
    BeforeValidator(before_validator_2),
    AfterValidator(after_validator_2),
    AfterValidator(after_validator_3),
    BeforeValidator(before_validator_3),
]

In [16]:
class Model(BaseModel):
    number: CustomType

In [17]:
print(f"Model(number=10): {Model(number=10)!r}")

before_validator_3
before_validator_2
before_validator_1
after_validator_1
after_validator_2
after_validator_3
Model(number=10): Model(number=10)


Outro exemplo de validadores com anotações.

Suponha um campo que é uma lista, de algum tipo, que só pode conter elementos **únicos** (o caso que o `conlist(unique_items=...)`
removido deixou em aberto na seção 5).

Queremos reutilizá-lo, então vamos implementá-lo com anotações.

Primeiro um tipo anotado só para inteiros; depois usamos a técnica de `TypeVar` para estender a qualquer tipo.

Primeiro o validador:

In [18]:
def are_elements_unique(values: list[Any]) -> list[Any]:
    unique_elements = []
    for value in values:
        if value in unique_elements:
            raise ValueError("elements must be unique")
        unique_elements.append(value)
    return values

Por que não usar um `set` para verificar se todos os elementos são únicos?

In [19]:
print(f"len(set([1, 2, 3, 4, 5])) == len([1, 2, 3, 4, 5]): {len(set([1, 2, 3, 4, 5])) == len([1, 2, 3, 4, 5])!r}")

len(set([1, 2, 3, 4, 5])) == len([1, 2, 3, 4, 5]): True


In [20]:
print(f"len(set([1, 1, 3, 4, 5])) == len([1, 1, 3, 4, 5]): {len(set([1, 1, 3, 4, 5])) == len([1, 1, 3, 4, 5])!r}")

len(set([1, 1, 3, 4, 5])) == len([1, 1, 3, 4, 5]): False


Porque sets só aceitam valores **hashable** — e talvez queiramos listas de tipos não-hashable (como dicts ou listas). Há formas mais
eficientes (a verificação acima é O(n²)), mas serve ao propósito aqui.

Agora o tipo anotado:

In [21]:
UniqueIntegerList = Annotated[list[int], AfterValidator(are_elements_unique)]

In [22]:
class Model(BaseModel):
    numbers: UniqueIntegerList = []

In [23]:
m = Model(numbers=(1, 2, 3, 4, 5))
print(f"m: {m!r}")

m: Model(numbers=[1, 2, 3, 4, 5])


In [24]:
try:
    Model(numbers=[1, 1, 2, 3])
except ValidationError as ex:
    print(f"{type(ex).__name__}: {ex}")

ValidationError: 1 validation error for Model
numbers
  Value error, elements must be unique [type=value_error, input_value=[1, 1, 2, 3], input_type=list]
    For further information visit https://errors.pydantic.dev/2.13/v/value_error


Estendendo para listas homogêneas de qualquer tipo:

In [25]:
from typing import TypeVar

In [26]:
T = TypeVar('T')

In [27]:
UniqueList = Annotated[list[T], AfterValidator(are_elements_unique)]

In [28]:
class Model(BaseModel):
    numbers: UniqueList[int] = []
    strings: UniqueList[str] = []

In [29]:
print(f'Model(numbers=[1, 2, 3], strings=["pyt", "hon"]): {Model(numbers=[1, 2, 3], strings=["pyt", "hon"])!r}')

Model(numbers=[1, 2, 3], strings=["pyt", "hon"]): Model(numbers=[1, 2, 3], strings=['pyt', 'hon'])


In [30]:
try:
    Model(numbers=[1, 1, 2])
except ValidationError as ex:
    print(f"{type(ex).__name__}: {ex}")

ValidationError: 1 validation error for Model
numbers
  Value error, elements must be unique [type=value_error, input_value=[1, 1, 2], input_type=list]
    For further information visit https://errors.pydantic.dev/2.13/v/value_error


In [31]:
try:
    Model(numbers=["a", 2, 3], strings=[1, "b"])
except ValidationError as ex:
    print(f"{type(ex).__name__}: {ex}")

ValidationError: 2 validation errors for Model
numbers.0
  Input should be a valid integer, unable to parse string as an integer [type=int_parsing, input_value='a', input_type=str]
    For further information visit https://errors.pydantic.dev/2.13/v/int_parsing
strings.0
  Input should be a valid string [type=string_type, input_value=1, input_type=int]
    For further information visit https://errors.pydantic.dev/2.13/v/string_type


Podemos ir além: exigir lista única, homogênea **e** com limites de tamanho — usando `Field`:

In [32]:
UniqueList = Annotated[
    list[T],
    Field(min_length=1, max_length=5),
    AfterValidator(are_elements_unique)
]

In [33]:
class Model(BaseModel):
    numbers: UniqueList[int] = []
    strings: UniqueList[str] = []

In [34]:
print(f'Model(numbers=[1, 2, 3], strings=["a", "b", "c"]): {Model(numbers=[1, 2, 3], strings=["a", "b", "c"])!r}')

Model(numbers=[1, 2, 3], strings=["a", "b", "c"]): Model(numbers=[1, 2, 3], strings=['a', 'b', 'c'])


In [35]:
try:
    Model(numbers=[], strings=list("python"))
except ValidationError as ex:
    print(f"{type(ex).__name__}: {ex}")

ValidationError: 2 validation errors for Model
numbers
  List should have at least 1 item after validation, not 0 [type=too_short, input_value=[], input_type=list]
    For further information visit https://errors.pydantic.dev/2.13/v/too_short
strings
  List should have at most 5 items after validation, not 6 [type=too_long, input_value=['p', 'y', 't', 'h', 'o', 'n'], input_type=list]
    For further information visit https://errors.pydantic.dev/2.13/v/too_long


Notou o problema com os defaults do modelo?

Agora há uma restrição de que as listas não podem ser vazias, mas os defaults são listas vazias (e, como defaults não são validados, isso
passa despercebido).

In [36]:
print(f"Model(): {Model()!r}")

Model(): Model(numbers=[], strings=[])


Como não há um default adequado aqui, removemos os defaults e tornamos os campos obrigatórios:

In [37]:
class Model(BaseModel):
    numbers: UniqueList[int]
    strings: UniqueList[str]

In [38]:
try:
    Model()
except ValidationError as ex:
    print(f"{type(ex).__name__}: {ex}")

ValidationError: 2 validation errors for Model
numbers
  Field required [type=missing, input_value={}, input_type=dict]
    For further information visit https://errors.pydantic.dev/2.13/v/missing
strings
  Field required [type=missing, input_value={}, input_type=dict]
    For further information visit https://errors.pydantic.dev/2.13/v/missing
